### 복습
- ratings_train.txt 데이터 로드
- document 컬럼의 결측치 제외
- 텍스트 정규화 (특수문자 제거, 2칸 이상의 공백 제외, 문자열 좌우 공백 제거)
- document의 중복 데이터를 제거
- 데이터프레임에서 상위 1000개만 사용
- texts -> 데이터프레임의 document컬럼의 values
- labels -> 데이터프레임의 label 컬럼의 vlaues
- 토큰화 -> Komoran
    - 선택하는 품사는 NNP, NNG, VV, VA, MAG, XR
    - 불용어 하다, 되다, 이다
- 단어 사전 생성
    - 단어들 중 최소 출현 횟수가 2회
    - 단어 사전에는 <PAD>, <UNK>을 제일 앞에 지정하여 단어사전 생성
- 토큰화 된 문서들을 단어 사전의 위치 값에 맞게 인코딩(enc_inputs)

In [14]:
import pandas as pd
import numpy as np
from konlpy.tag import Komoran
import re, collections

In [3]:
data = pd.read_csv("../data/ratings_train.txt", sep='\t')
data.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [4]:
data.dropna(inplace=True)

In [5]:
# 텍스트 정규화 함수
def nomalize(text):
    # text 매개변수에 들어오는 데이터?
        # df에 있는 document 컬럼의 vlaues ->> 리뷰 데이터
    # str(text) 사용하는 이유는?
        # 리뷰의 데이터가 문자가 아닌 경우 문자형으로 변경
    text = re.sub(r"[^각-힣0-9a-zA-Z\s\.]", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [7]:
data = data.applymap(nomalize)

C:\Users\abohv\AppData\Local\Temp\ipykernel_14068\2476581451.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data = data.applymap(nomalize)


In [8]:
data = data.loc[
    data['document'].str.len() > 1
]

In [9]:
data = data.drop_duplicates('document')

In [12]:
data = data.head(1000)

In [11]:
# 토큰화 함수를 생성
# 특정 품사들만 선택
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XR']
# 불용어
stop_word = ['하다', '되다', '이다', '것', '수','거']

# Komoram 객채를 생성
komoran  = Komoran()

def tokenize_komoran(text):
    tokens = []
    for word, pos in komoran.pos(text):
        # wrod : 단어
        # pos : 품사
        if pos in allow_pos and word not in  stop_word:
            tokens.append(word)  
    return tokens   

In [13]:
tokenize_sentence = [tokenize_komoran(val) for val in data['document'].values]
tokenize_sentence

[['더빙', '진짜', '짜증', '나', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없', '평점', '조정'],
 ['익살', '연기', '돋보이', '영화', '스파이더맨', '늙', '보이', '하', '커스틴 던스트', '너무나'],
 ['막', '걸음마', '떼', '초등학교', '학년', '용', '영화', '별', '반개', '아깝'],
 ['원작', '긴장감', '제대로', '살리'],
 ['반개',
  '아깝',
  '욕',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '이',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '이',
  '드라마',
  '족도',
  '없',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '없', '재미', '있', '안', '영화'],
 ['왜', '평점', '낮', '꽤', '보', '헐리우드', '화려', '너무', '길들이', '있'],
 ['짱', '진짜', '짱'],
 ['볼', '때', '눈물', '나서', '죽', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['울', '손들', '횡단보도', '건너', '때', '뛰쳐나오', '이범수', '연기', '드럽'],
 ['담백', '깔끔', '좋', '신문', '기사', '로만', '보다', '보', '자꾸', '잊어버리', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보',
  '영화',
  '장',
  '노',
  '재',
  '노',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['참',
  '사람',
  '웃기',
  '바스코',
  '이기',
  '락스',
  '코',
  '까',
  '고',
  '바비',
  '이기'

In [15]:
min_count=2
# 단어들의 출현 횟수를 dict의 형태로 생성
freq = collections.Counter(
    word for doc in tokenize_sentence for word in doc
)
freq

Counter({'영화': 365,
         '보': 261,
         '하': 135,
         '없': 114,
         '있': 87,
         '좋': 74,
         '진짜': 70,
         '정말': 67,
         '안': 66,
         '너무': 62,
         '연기': 58,
         '같': 57,
         '재밌': 57,
         '나오': 52,
         '만들': 51,
         '다': 48,
         '잘': 47,
         '최고': 46,
         '사람': 44,
         '왜': 43,
         '되': 40,
         '알': 38,
         '내용': 36,
         '때': 35,
         '감동': 34,
         '말': 33,
         '더': 33,
         '재미': 32,
         '재미없': 32,
         '아깝': 31,
         '배우': 31,
         '재미있': 30,
         '생각': 30,
         '시간': 29,
         '드라마': 28,
         '그냥': 28,
         '좀': 28,
         '나': 27,
         '평점': 27,
         '이': 27,
         '스토리': 27,
         '감독': 27,
         '쓰레기': 26,
         '작품': 25,
         '모르': 24,
         '완전': 23,
         '들': 23,
         '지루': 22,
         '보이': 21,
         '느낌': 21,
         '남': 20,
         '작': 20,
         '정도': 20,
     

In [16]:
vocab = ['<PAD>', '<UNK>'] + [
    word for word, cnt in freq.items() if cnt >= min_count
]
vocab

['<PAD>',
 '<UNK>',
 '더빙',
 '진짜',
 '짜증',
 '나',
 '목소리',
 '포스터',
 '초딩',
 '영화',
 '오버',
 '연기',
 '이야기',
 '솔직히',
 '재미',
 '없',
 '평점',
 '돋보이',
 '늙',
 '보이',
 '하',
 '너무나',
 '막',
 '떼',
 '초등학교',
 '학년',
 '용',
 '별',
 '반개',
 '아깝',
 '원작',
 '긴장감',
 '제대로',
 '살리',
 '욕',
 '나오',
 '생활',
 '이',
 '정말',
 '발로',
 '반복',
 '드라마',
 '못하',
 '사람',
 '액션',
 '있',
 '안',
 '왜',
 '낮',
 '꽤',
 '보',
 '헐리우드',
 '화려',
 '너무',
 '짱',
 '볼',
 '때',
 '눈물',
 '나서',
 '죽',
 '향수',
 '자극',
 '감성',
 '절제',
 '멜로',
 '울',
 '드럽',
 '담백',
 '깔끔',
 '좋',
 '기사',
 '보다',
 '자꾸',
 '취향',
 '극장',
 '장',
 '노',
 '재',
 '감동',
 '스토리',
 '어거지',
 '참',
 '웃기',
 '이기',
 '코',
 '까',
 '고',
 '깔',
 '그냥',
 '난',
 '이해',
 '뒤',
 '갈수록',
 '재미없',
 '이건',
 '깨알',
 '캐스팅',
 '내용',
 '구성',
 '잘',
 '러',
 '드',
 '위하',
 '착하',
 '절대',
 '뜻',
 '웃',
 '불',
 '능하',
 '지루',
 '같',
 '음식',
 '만찬',
 '넘',
 '별로',
 '평범',
 '수작',
 '드리',
 '주제',
 '중반',
 '다',
 '납득',
 '그렇',
 '꼭',
 '걸',
 '끄',
 '고추',
 '발',
 '센스',
 '연출력',
 '탁월',
 '90년대',
 '포스',
 '다시',
 '깨닫',
 '남',
 '꽃',
 '완전',
 '졸',
 '쓰레기',
 '시간',
 '재밌',
 '별점',
 '이리',
 '기대',
 '죄인'

In [17]:
word_list = []
for word, cnt in freq.items():
    # print(word)
    # print(cnt)
    # break
    if cnt >= min_count:
        word_list.append(word)
# for key in freq:
#     print(key)
#     print(freq[key])
#     break
word_list

['더빙',
 '진짜',
 '짜증',
 '나',
 '목소리',
 '포스터',
 '초딩',
 '영화',
 '오버',
 '연기',
 '이야기',
 '솔직히',
 '재미',
 '없',
 '평점',
 '돋보이',
 '늙',
 '보이',
 '하',
 '너무나',
 '막',
 '떼',
 '초등학교',
 '학년',
 '용',
 '별',
 '반개',
 '아깝',
 '원작',
 '긴장감',
 '제대로',
 '살리',
 '욕',
 '나오',
 '생활',
 '이',
 '정말',
 '발로',
 '반복',
 '드라마',
 '못하',
 '사람',
 '액션',
 '있',
 '안',
 '왜',
 '낮',
 '꽤',
 '보',
 '헐리우드',
 '화려',
 '너무',
 '짱',
 '볼',
 '때',
 '눈물',
 '나서',
 '죽',
 '향수',
 '자극',
 '감성',
 '절제',
 '멜로',
 '울',
 '드럽',
 '담백',
 '깔끔',
 '좋',
 '기사',
 '보다',
 '자꾸',
 '취향',
 '극장',
 '장',
 '노',
 '재',
 '감동',
 '스토리',
 '어거지',
 '참',
 '웃기',
 '이기',
 '코',
 '까',
 '고',
 '깔',
 '그냥',
 '난',
 '이해',
 '뒤',
 '갈수록',
 '재미없',
 '이건',
 '깨알',
 '캐스팅',
 '내용',
 '구성',
 '잘',
 '러',
 '드',
 '위하',
 '착하',
 '절대',
 '뜻',
 '웃',
 '불',
 '능하',
 '지루',
 '같',
 '음식',
 '만찬',
 '넘',
 '별로',
 '평범',
 '수작',
 '드리',
 '주제',
 '중반',
 '다',
 '납득',
 '그렇',
 '꼭',
 '걸',
 '끄',
 '고추',
 '발',
 '센스',
 '연출력',
 '탁월',
 '90년대',
 '포스',
 '다시',
 '깨닫',
 '남',
 '꽃',
 '완전',
 '졸',
 '쓰레기',
 '시간',
 '재밌',
 '별점',
 '이리',
 '기대',
 '죄인',
 '아직',
 '인생',
 '최고

In [18]:
stoi = {
    word : idx for idx, word in enumerate(vocab)
}
stoi

{'<PAD>': 0,
 '<UNK>': 1,
 '더빙': 2,
 '진짜': 3,
 '짜증': 4,
 '나': 5,
 '목소리': 6,
 '포스터': 7,
 '초딩': 8,
 '영화': 9,
 '오버': 10,
 '연기': 11,
 '이야기': 12,
 '솔직히': 13,
 '재미': 14,
 '없': 15,
 '평점': 16,
 '돋보이': 17,
 '늙': 18,
 '보이': 19,
 '하': 20,
 '너무나': 21,
 '막': 22,
 '떼': 23,
 '초등학교': 24,
 '학년': 25,
 '용': 26,
 '별': 27,
 '반개': 28,
 '아깝': 29,
 '원작': 30,
 '긴장감': 31,
 '제대로': 32,
 '살리': 33,
 '욕': 34,
 '나오': 35,
 '생활': 36,
 '이': 37,
 '정말': 38,
 '발로': 39,
 '반복': 40,
 '드라마': 41,
 '못하': 42,
 '사람': 43,
 '액션': 44,
 '있': 45,
 '안': 46,
 '왜': 47,
 '낮': 48,
 '꽤': 49,
 '보': 50,
 '헐리우드': 51,
 '화려': 52,
 '너무': 53,
 '짱': 54,
 '볼': 55,
 '때': 56,
 '눈물': 57,
 '나서': 58,
 '죽': 59,
 '향수': 60,
 '자극': 61,
 '감성': 62,
 '절제': 63,
 '멜로': 64,
 '울': 65,
 '드럽': 66,
 '담백': 67,
 '깔끔': 68,
 '좋': 69,
 '기사': 70,
 '보다': 71,
 '자꾸': 72,
 '취향': 73,
 '극장': 74,
 '장': 75,
 '노': 76,
 '재': 77,
 '감동': 78,
 '스토리': 79,
 '어거지': 80,
 '참': 81,
 '웃기': 82,
 '이기': 83,
 '코': 84,
 '까': 85,
 '고': 86,
 '깔': 87,
 '그냥': 88,
 '난': 89,
 '이해': 90,
 '뒤': 91,
 '갈수록': 9

In [19]:
def encode(words):
    result = [stoi.get(word, stoi['<UNK>']) for word in words]
    return result

enc_inputs = [encode(doc) for doc in tokenize_sentence]
enc_inputs

[[2, 3, 4, 5, 6],
 [7, 8, 9, 10, 11],
 [],
 [1, 12, 13, 14, 15, 16, 1],
 [1, 11, 17, 9, 1, 18, 19, 20, 1, 21],
 [22, 1, 23, 24, 25, 26, 9, 27, 28, 29],
 [30, 31, 32, 33],
 [28,
  29,
  34,
  35,
  1,
  1,
  11,
  36,
  37,
  38,
  39,
  1,
  1,
  40,
  40,
  37,
  41,
  1,
  15,
  11,
  42,
  43,
  1],
 [44, 15, 14, 45, 46, 9],
 [47, 16, 48, 49, 50, 51, 52, 53, 1, 45],
 [54, 3, 54],
 [55, 56, 57, 58, 59, 60, 61, 1, 62, 63, 64, 1],
 [65, 1, 1, 1, 56, 1, 1, 11, 66],
 [67, 68, 69, 1, 70, 1, 71, 50, 72, 1, 43],
 [73, 1, 3, 74, 50, 9, 75, 76, 77, 76, 78, 79, 80, 78, 80],
 [1, 1],
 [81, 43, 82, 1, 83, 1, 84, 85, 86, 1, 83, 1, 87, 88, 85, 1, 89, 19],
 [1, 1, 90, 47, 91, 92, 93],
 [94, 38, 95, 96, 1, 1, 97, 98, 99, 1, 100, 1, 95, 101],
 [1, 102, 1, 103, 104],
 [1, 105, 45, 88, 1, 1, 1, 9, 104],
 [50, 106, 107, 108],
 [93,
  109,
  110,
  111,
  9,
  112,
  113,
  112,
  12,
  45,
  111,
  50,
  14,
  45,
  94,
  50,
  15,
  111,
  114,
  46,
  35,
  1,
  1,
  114,
  46,
  35],
 [104, 115, 9, 1